# Assignment 8 — Image Segmentation

Each problem is solved in two ways: **Style 1** uses OpenCV/NumPy built-in operations, while **Style 2** implements the algorithm with arrays, loops, and a queue. Run the cells from this notebook folder.


## Style 1 — Built-in library functions

These reference implementations use OpenCV thresholding, adaptive thresholding, histogram functions, and flood filling.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LAB_DIR = Path.cwd()

def read_gray(name):
    image = cv2.imread(str(LAB_DIR / name), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"Missing image: {LAB_DIR / name}")
    return image

def iterative_threshold_cv(image):
    threshold = float(image.mean())
    while True:
        _, result = cv2.threshold(image, threshold, 255, cv2.THRESH_BINARY)
        lower, upper = image[result == 0], image[result == 255]
        new_threshold = (float(lower.mean()) if lower.size else 0.0) + (float(upper.mean()) if upper.size else 0.0)
        count = int(lower.size > 0) + int(upper.size > 0)
        new_threshold /= max(count, 1)
        if abs(new_threshold - threshold) < 0.5:
            return new_threshold
        threshold = new_threshold

def threshold_library(image, threshold):
    return cv2.threshold(image, threshold, 255, cv2.THRESH_BINARY)[1]

def local_threshold_library(image):
    return cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 5)

def region_grow_library(image, seed, tolerance=50):
    work = image.copy()
    mask = np.zeros((image.shape[0] + 2, image.shape[1] + 2), np.uint8)
    cv2.floodFill(work, mask, (seed[1], seed[0]), 255, loDiff=tolerance, upDiff=tolerance, flags=4)
    return mask[1:-1, 1:-1] > 0

horse = read_gray("noisy_horse.png")
T = iterative_threshold_cv(horse)
print("Built-in iterative threshold:", T)
plt.figure(figsize=(10, 3)); plt.subplot(131); plt.imshow(horse, cmap="gray"); plt.title("Input"); plt.axis("off")
plt.subplot(132); plt.imshow(threshold_library(horse, T), cmap="gray"); plt.title("Global threshold"); plt.axis("off")
plt.subplot(133); plt.imshow(local_threshold_library(horse), cmap="gray"); plt.title("Local threshold"); plt.axis("off"); plt.show()
brain = read_gray("brain.png")
seed = (200, 100)
print("Built-in grown pixels:", int(region_grow_library(brain, seed).sum()))


## Style 2 — DSA/manual implementation


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LAB_DIR = Path.cwd()

def read_gray(name):
    image = cv2.imread(str(LAB_DIR / name), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"Missing image: {LAB_DIR / name}")
    return image

def average_filter(image, size=5):
    pad = size // 2
    padded = np.pad(image, pad, mode="edge")
    result = np.zeros_like(image, dtype=np.uint8)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            result[r, c] = np.clip(np.rint(padded[r:r+size, c:c+size].mean()), 0, 255)
    return result

def basic_threshold(image, tolerance=0.5):
    threshold = float(image.mean())
    while True:
        lower = image[image < threshold]
        upper = image[image >= threshold]
        new_threshold = (lower.mean() if lower.size else 0 + upper.mean() if upper.size else 0) / (int(lower.size > 0) + int(upper.size > 0))
        if abs(new_threshold - threshold) < tolerance:
            return new_threshold
        threshold = new_threshold

def binary_threshold(image, threshold):
    return (image >= threshold).astype(np.uint8) * 255

def semi_threshold(image, threshold):
    return np.where(image >= threshold, image, 0).astype(np.uint8)

def histogram(image):
    return np.bincount(image.ravel(), minlength=256)

def local_peaks(counts, radius=20):
    peaks = []
    for z in range(256):
        lo, hi = max(0, z-radius), min(255, z+radius)
        if counts[z] == counts[lo:hi+1].max() and counts[z] > 0:
            peaks.append(z)
    return peaks

def mode_pair(counts, p, q):
    valley = p + int(np.argmin(counts[p:q+1]))
    n = counts / counts.sum()
    s1 = (n[p] - n[valley]) / max(valley-p, 1)
    s2 = (n[q] - n[valley]) / max(q-valley, 1)
    kind = "bimodal" if s1 >= 5e-5 and s2 >= 5e-5 else ("peak and shoulder" if max(s1, s2) >= 5e-5 else "flat")
    return valley, s1, s2, kind

horse = read_gray("noisy_horse.png")
horse_smooth = average_filter(horse)
T_horse, T_smooth = basic_threshold(horse), basic_threshold(horse_smooth)
print(f"noisy_horse threshold: {T_horse:.2f}; smoothed threshold: {T_smooth:.2f}")
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, image, title in zip(axes.flat[:3], [horse, horse_smooth, histogram(horse_smooth)], ["f", "smoothed f", "smoothed histogram"]):
    ax.imshow(image, cmap="gray") if image.ndim == 2 else ax.plot(image); ax.set_title(title); ax.axis("off")
for ax, image, title in zip(axes.flat[3:], [binary_threshold(horse, T_horse), binary_threshold(horse_smooth, T_smooth), semi_threshold(horse_smooth, T_smooth)], ["threshold f", "threshold smoothed f", "semi-threshold smoothed f"]):
    ax.imshow(image, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.tight_layout()

camera = average_filter(read_gray("cameraman.png"))
counts = histogram(camera); peaks = local_peaks(counts)
print("Detected peaks:", peaks)
if len(peaks) >= 3:
    valleys = [mode_pair(counts, a, b) for a, b in zip(peaks[:-1], peaks[1:])]
    thresholds = [v[0] for v in valleys[:2]]
    print("Valleys (m, s1, s2, class):", valleys)
    multi = np.digitize(camera, thresholds).astype(np.uint8) * 127
    band = np.where((camera >= thresholds[0]) & (camera < thresholds[1]), 255, 0).astype(np.uint8)
    print("D1/D2/D3 pixels:", [int(np.count_nonzero(camera < thresholds[0])), int(np.count_nonzero((camera >= thresholds[0]) & (camera < thresholds[1]))), int(np.count_nonzero(camera >= thresholds[1]))])
    fig, ax = plt.subplots(1, 3, figsize=(15, 4)); ax[0].plot(counts); ax[0].scatter(peaks, counts[peaks], c="r"); ax[0].scatter(thresholds, counts[thresholds], c="g"); ax[1].imshow(multi, cmap="gray"); ax[2].imshow(band, cmap="gray"); plt.show()

shaded = read_gray("shaded_disks.png")
print(f"shaded global threshold: {basic_threshold(shaded):.2f}")
for index, block in enumerate(np.array_split(shaded, 2, axis=0)):
    for col, tile in enumerate(np.array_split(block, 3, axis=1), 1):
        peaks = local_peaks(histogram(tile)); pair = mode_pair(histogram(tile), peaks[0], peaks[1]) if len(peaks) >= 2 else (None, None, None, "flat")
        print(f"B{index*3+col}: threshold={basic_threshold(tile):.2f}, peaks={peaks[:2]}, troublesome={len(peaks)<2 or pair[3] != 'bimodal'}")

brain = read_gray("brain.png")
def grow(image, seed, tolerance=50):
    region = np.zeros(image.shape, bool); region[seed] = True; stack = [seed]
    while stack:
        r, c = stack.pop()
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                nr, nc = r+dr, c+dc
                if 0 <= nr < image.shape[0] and 0 <= nc < image.shape[1] and not region[nr, nc] and abs(int(image[r, c])-int(image[nr, nc])) <= tolerance:
                    region[nr, nc] = True; stack.append((nr, nc))
    return region
for seed in [(200, 100), (300, 200)]:
    region = grow(brain, seed); print(f"Seed {seed}: {int(region.sum())} pixels")


## Comparison

The built-in cell demonstrates the production API; the following cell contains the same segmentation ideas without calling the corresponding thresholding or flood-fill helper.
